# Fake News Detection — Comparative Experiments
## CDS525 Group Project

**五组对比实验全流程 Notebook**

| 实验组 | 方法 | 预计耗时 (H100) |
|--------|------|:----------------:|
| 组1 | RNN 架构消融 (LSTM/BiLSTM/GRU + 注意力 + 池化) | ~1h |
| 组2 | LLM 微调 (BERT/RoBERTa/DistilBERT) | ~1h |
| 组3 | 多智能体网络搜索验证 | ~5-15min |
| 组4 | 知识图谱 + 强化学习 | ~5-10min |
| 组5 | MMDFND 复现 + 六大创新 | ~5h |

**运行前**: Runtime → Change runtime type → **GPU (推荐 A100/H100)**

---
## 0. 环境配置

In [ ]:
# ============================================================
# 0.1 检查 GPU
# ============================================================
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB)")
else:
    print("WARNING: No GPU! Runtime -> Change runtime type -> GPU")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ============================================================
# 0.2 挂载 Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# 检查项目目录和数据文件
PROJECT_DIR = '/content/drive/MyDrive/fakenews-detector'
print(f"项目目录: {PROJECT_DIR}")
print(f"  存在: {os.path.isdir(PROJECT_DIR)}")

for name in ['fakenews 2.csv', 'News _dataset/Fake.csv', 'News _dataset/True.csv']:
    p = os.path.join(PROJECT_DIR, name)
    if os.path.exists(p):
        print(f"  OK  {name} ({os.path.getsize(p)/1e6:.1f} MB)")
    else:
        print(f"  MISSING  {p}")

In [ ]:
# ============================================================
# 0.3 复制项目到本地磁盘 (Drive I/O 太慢，必须复制到本地)
# ============================================================
import shutil, sys, time

DRIVE_PROJECT = '/content/drive/MyDrive/fakenews-detector'
PROJECT_DIR = '/content/fakenews-detector'

if not os.path.exists(PROJECT_DIR):
    print(f"复制项目到本地 (首次约 1-3 分钟)...")
    t0 = time.time()

    # 只复制代码和小文件，跳过大文件和不需要的目录
    def ignore_large(dir, files):
        skip = set()
        for f in files:
            full = os.path.join(dir, f)
            if os.path.isfile(full) and os.path.getsize(full) > 200_000_000:
                skip.add(f)  # 跳过 >200MB 的文件 (GloVe, zip等)
            if f in ('.git', '__pycache__', 'DITFEND', 'L-Defense_EFND',
                      'glove.6B.zip', 'outputs'):
                skip.add(f)
            # 跳过保存的网页资源
            if f.endswith('_files') or f.endswith('.html'):
                skip.add(f)
        return skip

    shutil.copytree(DRIVE_PROJECT, PROJECT_DIR, ignore=ignore_large)
    print(f"代码复制完成 ({time.time()-t0:.1f}s)")
else:
    print(f"项目已存在: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

print(f"工作目录: {os.getcwd()}")
print(f"experiments/ : {os.path.isdir('experiments')}")
print(f"src/ : {os.path.isdir('src')}")

In [ ]:
# ============================================================
# 0.4 安装依赖
# ============================================================
!pip install -q torch pandas numpy scikit-learn matplotlib seaborn nltk tqdm
!pip install -q transformers accelerate timm

# 可选: 组3/组4 需要的 API 客户端
# !pip install -q openai tavily-python

import nltk
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print("依赖安装完成")

In [ ]:
# ============================================================
# 0.5 复制数据文件到本地项目目录
# ============================================================
import shutil, time

DRIVE_PROJECT = '/content/drive/MyDrive/fakenews-detector'
os.chdir(PROJECT_DIR)

# 需要复制的大数据文件 (代码已在 0.3 复制，这里只处理数据)
data_files = [
    ('fakenews 2.csv', os.path.join(DRIVE_PROJECT, 'fakenews 2.csv'),
     os.path.join(PROJECT_DIR, 'fakenews 2.csv')),
    ('Fake.csv', os.path.join(DRIVE_PROJECT, 'News _dataset', 'Fake.csv'),
     os.path.join(PROJECT_DIR, 'News _dataset', 'Fake.csv')),
    ('True.csv', os.path.join(DRIVE_PROJECT, 'News _dataset', 'True.csv'),
     os.path.join(PROJECT_DIR, 'News _dataset', 'True.csv')),
]

os.makedirs(os.path.join(PROJECT_DIR, 'News _dataset'), exist_ok=True)

for name, src, dst in data_files:
    if os.path.exists(dst):
        print(f"  OK  {name} ({os.path.getsize(dst)/1e6:.1f} MB)")
    elif os.path.exists(src):
        t0 = time.time()
        shutil.copy2(src, dst)
        print(f"  Copied {name} ({os.path.getsize(dst)/1e6:.1f} MB, {time.time()-t0:.1f}s)")
    else:
        print(f"  MISSING: {src}")

# GloVe: 优先从 Drive 复制，否则下载
glove_dst = os.path.join(PROJECT_DIR, 'glove.6B.100d.txt')
glove_src = os.path.join(DRIVE_PROJECT, 'glove.6B.100d.txt')

if os.path.exists(glove_dst):
    print(f"  OK  glove.6B.100d.txt ({os.path.getsize(glove_dst)/1e6:.0f} MB)")
elif os.path.exists(glove_src):
    print("  Copying GloVe from Drive (347 MB, ~1-2 min)...")
    t0 = time.time()
    shutil.copy2(glove_src, glove_dst)
    print(f"  GloVe copied ({time.time()-t0:.1f}s)")
else:
    print("  Downloading GloVe (347 MB)...")
    !wget -q https://nlp.stanford.edu/data/glove.6B.zip -O /tmp/glove.6B.zip
    !unzip -qo /tmp/glove.6B.zip glove.6B.100d.txt -d "{PROJECT_DIR}"
    print("  GloVe downloaded")

print(f"\n所有数据已在本地磁盘: {PROJECT_DIR}")
!du -sh "{PROJECT_DIR}"

In [ ]:
# ============================================================
# 0.6 验证项目结构
# ============================================================
required = [
    'src/data_utils.py', 'src/model.py', 'src/trainer.py',
    'experiments/config.py', 'experiments/runner.py', 'experiments/metrics.py',
    'experiments/group1_rnn/models.py', 'experiments/group1_rnn/run_group1.py',
    'experiments/group2_llm/models.py', 'experiments/group2_llm/run_group2.py',
    'experiments/group3_multiagent/agents.py',
    'experiments/group4_kg_rl/rl_agent.py',
    'experiments/evaluate_all.py',
]
for f in required:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    print(f"  [{status}] {f}")

---
## 1. 实验组 1：RNN 架构消融

系统性测试 4 种 RNN × 4 种注意力 × 4 种池化的组合，加上超参数和嵌入消融。

- 完整模式: 33 组实验，~1 小时
- 快速模式 (`--quick`): 4 组实验，~10 分钟

In [ ]:
# ============================================================
# 1.1 运行 RNN 消融实验
# ============================================================
%cd /content/fakenews-detector

# 快速模式: 仅跑少量组合验证流程
!python -m experiments.group1_rnn.run_group1 --quick

# 完整模式: 取消下面的注释
# !python -m experiments.group1_rnn.run_group1

In [ ]:
# ============================================================
# 1.2 查看组1结果
# ============================================================
import pandas as pd
from IPython.display import display

g1_csv = 'outputs/group1_rnn/results.csv'
if os.path.exists(g1_csv):
    df = pd.read_csv(g1_csv)
    display(df[['experiment_name', 'accuracy', 'f1', 'auc_roc', 'param_count']]
            .sort_values('f1', ascending=False))
else:
    print("组1尚未运行")

from IPython.display import Image as IPImage
img_path = 'outputs/group1_rnn/group1_comparison.png'
if os.path.exists(img_path):
    display(IPImage(img_path, width=800))

---
## 2. 实验组 2：LLM 预训练模型微调

对比 DistilBERT / BERT / RoBERTa 在三种冻结策略下的表现。

- 完整模式: 9 组实验，~1 小时
- 快速模式: 仅 DistilBERT 全量微调，~10 分钟

In [ ]:
# ============================================================
# 2.1 运行 LLM 微调实验
# ============================================================
%cd /content/fakenews-detector

# 快速模式: 仅 DistilBERT + 全量微调
!python -m experiments.group2_llm.run_group2 --quick

# 完整模式:
# !python -m experiments.group2_llm.run_group2

In [ ]:
# ============================================================
# 2.2 查看组2结果
# ============================================================
g2_csv = 'outputs/group2_llm/results.csv'
if os.path.exists(g2_csv):
    df = pd.read_csv(g2_csv)
    display(df[['experiment_name', 'accuracy', 'f1', 'auc_roc', 'param_count']]
            .sort_values('f1', ascending=False))
else:
    print("组2尚未运行")

img_path = 'outputs/group2_llm/group2_comparison.png'
if os.path.exists(img_path):
    display(IPImage(img_path, width=800))

---
## 3. 实验组 3：多智能体网络搜索验证

通过 Claim 提取 → 网络搜索 → NLI 评分 → 裁判 的管道验证新闻真伪。

- **需要 API Key**: 设置 `OPENAI_API_KEY` 和 `TAVILY_API_KEY`
- **无 API 模式**: 使用启发式提取 + 模拟搜索，仍可运行

In [ ]:
# ============================================================
# 3.0 (可选) 设置 API Key
# ============================================================
# 如果有 API Key，取消下面的注释并填入

# import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'      # OpenAI API Key
# os.environ['TAVILY_API_KEY'] = 'tvly-...'     # Tavily Search API Key

# 或者使用 Colab Secrets (更安全):
# from google.colab import userdata
# os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
# os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')

In [ ]:
# ============================================================
# 3.1 运行多智能体验证
# ============================================================
%cd /content/fakenews-detector

# 快速模式: 20 条样本
!python -m experiments.group3_multiagent.run_group3 --quick

# 完整模式 (100 条):
# !python -m experiments.group3_multiagent.run_group3 --num-samples 100

# 使用 LLM 裁判 (需要 OPENAI_API_KEY):
# !python -m experiments.group3_multiagent.run_group3 --judge-mode llm --num-samples 100

In [ ]:
# ============================================================
# 3.2 查看组3结果
# ============================================================
g3_csv = 'outputs/group3_multiagent/results.csv'
if os.path.exists(g3_csv):
    df = pd.read_csv(g3_csv)
    display(df[['experiment_name', 'accuracy', 'precision', 'recall', 'f1']])
else:
    print("组3尚未运行")

---
## 4. 实验组 4：知识图谱 + 强化学习

- **元分类器**: 融合模型置信度 + KG 常识分数进行分类
- **DQN 路由**: 学习最优验证策略（查 KG / 搜网络 / 直接判定）

In [ ]:
# ============================================================
# 4.1 运行 KG + RL 实验
# ============================================================
%cd /content/fakenews-detector

# 快速模式: 30 条样本
!python -m experiments.group4_kg_rl.run_group4 --quick

# 完整模式:
# !python -m experiments.group4_kg_rl.run_group4 --num-samples 200

# 仅元分类器:
# !python -m experiments.group4_kg_rl.run_group4 --mode meta

# 仅 DQN:
# !python -m experiments.group4_kg_rl.run_group4 --mode dqn

In [ ]:
# ============================================================
# 4.2 查看组4结果
# ============================================================
g4_csv = 'outputs/group4_kg_rl/results.csv'
if os.path.exists(g4_csv):
    df = pd.read_csv(g4_csv)
    display(df[['experiment_name', 'accuracy', 'f1']])
else:
    print("组4尚未运行")

---
## 5. 实验组 5：MMDFND 复现与创新（可选）

**注意**: 此实验使用中文微博数据集 + 多模态模型，需要额外下载数据和预训练模型。

如果不需要运行组5，可以跳过此部分直接到第6步。

In [ ]:
# ============================================================
# 5.1 检查 MMDFND 环境
# ============================================================
%cd /content/fakenews-detector
!python -m experiments.group5_mmdfnd.run_group5 --check-env

In [ ]:
# ============================================================
# 5.2 查看配置指引
# ============================================================
%cd /content/fakenews-detector
!python -m experiments.group5_mmdfnd.run_group5 --setup

In [ ]:
# ============================================================
# 5.3 运行 MMDFND 训练 (需要先按指引准备数据)
# ============================================================
%cd /content/fakenews-detector
# 取消注释运行:
# !python -m experiments.group5_mmdfnd.run_group5 --train

---
## 6. 全局评估与对比

汇总所有已完成实验的结果，生成跨组对比图表和分析报告。

In [ ]:
# ============================================================
# 6.1 运行全局评估
# ============================================================
%cd /content/fakenews-detector
!python -m experiments.evaluate_all --output-dir outputs/evaluation

In [ ]:
# ============================================================
# 6.2 展示对比图表
# ============================================================
import glob
from IPython.display import Image as IPImage, display

eval_dir = 'outputs/evaluation'
if os.path.exists(eval_dir):
    for img in sorted(glob.glob(os.path.join(eval_dir, '*.png'))):
        print(f"\n--- {os.path.basename(img)} ---")
        display(IPImage(img, width=900))
else:
    print("请先运行上方的全局评估")

In [ ]:
# ============================================================
# 6.3 汇总所有结果表格
# ============================================================
import pandas as pd
from IPython.display import display

all_results = []
for csv_path in glob.glob('outputs/*/results.csv'):
    df = pd.read_csv(csv_path)
    all_results.append(df)

if all_results:
    combined = pd.concat(all_results, ignore_index=True)
    cols = ['group', 'experiment_name', 'accuracy', 'precision', 'recall', 'f1', 'auc_roc']
    available_cols = [c for c in cols if c in combined.columns]
    display(combined[available_cols].sort_values('f1', ascending=False))
    print(f"\n共 {len(combined)} 组实验结果")
else:
    print("暂无实验结果，请先运行上方实验")

---
## 7. 保存结果到 Google Drive

In [ ]:
# ============================================================
# 7.1 复制结果到 Google Drive (持久保存)
# ============================================================
import shutil

SAVE_DIR = '/content/drive/MyDrive/fakenews/experiment_results'
os.makedirs(SAVE_DIR, exist_ok=True)

src_outputs = os.path.join(PROJECT_DIR, 'outputs')
if os.path.exists(src_outputs):
    for group_dir in os.listdir(src_outputs):
        src = os.path.join(src_outputs, group_dir)
        dst = os.path.join(SAVE_DIR, group_dir)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print(f"  Saved: {group_dir}/")
    print(f"\n所有结果已保存到: {SAVE_DIR}")
else:
    print("outputs/ 目录不存在，请先运行实验")

---
## 快速参考

### 单独运行某组实验

```python
# 组1 (RNN 消融)
!python -m experiments.group1_rnn.run_group1           # 完整
!python -m experiments.group1_rnn.run_group1 --quick   # 快速

# 组2 (LLM 微调)
!python -m experiments.group2_llm.run_group2           # 完整
!python -m experiments.group2_llm.run_group2 --quick   # 快速

# 组3 (多智能体)
!python -m experiments.group3_multiagent.run_group3 --num-samples 100
!python -m experiments.group3_multiagent.run_group3 --quick

# 组4 (KG + RL)
!python -m experiments.group4_kg_rl.run_group4 --mode both
!python -m experiments.group4_kg_rl.run_group4 --quick

# 组5 (MMDFND)
!python -m experiments.group5_mmdfnd.run_group5 --check-env
!python -m experiments.group5_mmdfnd.run_group5 --train

# 全局评估
!python -m experiments.evaluate_all --output-dir outputs/evaluation
```

### 数据准备检查清单

| 文件 | Google Drive 路径 | 组1-4 必需 | 组5 必需 |
|------|-------------------|:----------:|:--------:|
| `fakenews 2.csv` | `My Drive/fakenews/` | Yes | No |
| `Fake.csv` + `True.csv` | `My Drive/fakenews/News _dataset/` | Yes | No |
| `glove.6B.100d.txt` | `My Drive/fakenews/` (或自动下载) | 组1 | No |
| 微博数据集 | MMDFND 论文提供 | No | Yes |